In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("E-Commerce Sales Analytics")
print("Python environment is ready!")

E-Commerce Sales Analytics
Python environment is ready!


In [2]:
import pandas as pd
import os

# Lokasi folder dataset
data_path = "../data/raw"

# Membaca semua file CSV
customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))
orders = pd.read_csv(os.path.join(data_path, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv"))
payments = pd.read_csv(os.path.join(data_path, "olist_order_payments_dataset.csv"))
reviews = pd.read_csv(os.path.join(data_path, "olist_order_reviews_dataset.csv"))
products = pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(data_path, "olist_sellers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(data_path, "olist_geolocation_dataset.csv"))
category_translation = pd.read_csv(os.path.join(data_path, "product_category_name_translation.csv"))

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
# Menggabungkan semua dataset ke dalam dictionary
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

# Menampilkan jumlah baris dan kolom
dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

dataset_summary

,Dataset,Rows,Columns
0,Customers,99441,5
1,Orders,99441,8
2,Order Items,112650,7
3,Payments,103886,5
4,Reviews,99224,7
5,Products,32951,9
6,Sellers,3095,4
7,Geolocation,1000163,5
8,Category Translation,71,2


In [4]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print(df.columns.tolist())


===== Customers =====
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

===== Orders =====
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

===== Order Items =====
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

===== Payments =====
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

===== Reviews =====
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

===== Products =====
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

===== Sellers =====
['seller_id', 's

In [5]:
# Memeriksa missing values pada seluruh dataset

missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()
        missing_percentage = (missing_count / len(df)) * 100

        missing_summary.append({
            "Dataset": name,
            "Column": column,
            "Missing Values": missing_count,
            "Missing (%)": round(missing_percentage, 2)
        })

missing_df = pd.DataFrame(missing_summary)

# Menampilkan hanya kolom yang memiliki missing values
missing_df[missing_df["Missing Values"] > 0].sort_values(
    by="Missing Values",
    ascending=False
)

,Dataset,Column,Missing Values,Missing (%)
28,Reviews,review_comment_title,87656,88.34
29,Reviews,review_comment_message,58247,58.70
11,Orders,order_delivered_customer_date,2965,2.98
10,Orders,order_delivered_carrier_date,1783,1.79
33,Products,product_category_name,610,1.85
35,Products,product_description_lenght,610,1.85
34,Products,product_name_lenght,610,1.85
36,Products,product_photos_qty,610,1.85
9,Orders,order_approved_at,160,0.16
37,Products,product_weight_g,2,0.01


In [6]:
# Menghitung jumlah setiap status pesanan

order_status_summary = orders["order_status"].value_counts().reset_index()

order_status_summary.columns = ["Order Status", "Total Orders"]

order_status_summary

,Order Status,Total Orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [7]:
# Membuat indikator missing values pada kolom tanggal
orders["missing_approved"] = orders["order_approved_at"].isnull()
orders["missing_carrier"] = orders["order_delivered_carrier_date"].isnull()
orders["missing_delivered"] = orders["order_delivered_customer_date"].isnull()

# Mengelompokkan berdasarkan status pesanan
status_missing_analysis = orders.groupby("order_status").agg(
    total_orders=("order_id", "count"),
    missing_approved=("missing_approved", "sum"),
    missing_carrier=("missing_carrier", "sum"),
    missing_delivered=("missing_delivered", "sum")
).reset_index()

status_missing_analysis

,order_status,total_orders,missing_approved,missing_carrier,missing_delivered
0,approved,2,0,2,2
1,canceled,625,141,550,619
2,created,5,5,5,5
3,delivered,96478,14,2,8
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,shipped,1107,0,0,1107
7,unavailable,609,0,609,609


In [8]:
print(status_missing_analysis.to_string(index=False))

print("\nTotal missing values:")
print(orders[[
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]].isnull().sum())

order_status  total_orders  missing_approved  missing_carrier  missing_delivered
    approved             2                 0                2                  2
    canceled           625               141              550                619
     created             5                 5                5                  5
   delivered         96478                14                2                  8
    invoiced           314                 0              314                314
  processing           301                 0              301                301
     shipped          1107                 0                0               1107
 unavailable           609                 0              609                609

Total missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


In [9]:
# Memeriksa jumlah duplicate records pada setiap dataset

duplicate_summary = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": name,
        "Duplicate Rows": duplicate_count,
        "Duplicate (%)": round(
            (duplicate_count / len(df)) * 100, 2
        )
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df

,Dataset,Duplicate Rows,Duplicate (%)
0,Customers,0,0.00
1,Orders,0,0.00
2,Order Items,0,0.00
3,Payments,0,0.00
4,Reviews,0,0.00
5,Products,0,0.00
6,Sellers,0,0.00
7,Geolocation,261831,26.18
8,Category Translation,0,0.00


In [10]:
# Memeriksa tipe data seluruh dataset

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print(df.dtypes)


===== Customers =====
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

===== Orders =====
order_id                          str
customer_id                       str
order_status                      str
order_purchase_timestamp          str
order_approved_at                 str
order_delivered_carrier_date      str
order_delivered_customer_date     str
order_estimated_delivery_date     str
missing_approved                 bool
missing_carrier                  bool
missing_delivered                bool
dtype: object

===== Order Items =====
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

===== Payments =====
order_id                    str
payment_sequential        int64

In [11]:
# Memeriksa statistik deskriptif seluruh dataset

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print(df.describe().T)


===== Customers =====
                            count          mean           std     min  \
customer_zip_code_prefix  99441.0  35137.474583  29797.938996  1003.0   

                              25%      50%      75%      max  
customer_zip_code_prefix  11347.0  24416.0  58900.0  99990.0  

===== Orders =====
                               count unique                               top  \
order_id                       99441  99441  e481f51cbdc54678b7cc49136f2d6af7   
customer_id                    99441  99441  9ef432eb6251297304e76186b10a928d   
order_status                   99441      8                         delivered   
order_purchase_timestamp       99441  98875               2018-03-31 15:08:21   
order_approved_at              99281  90733               2018-02-27 04:31:10   
order_delivered_carrier_date   97658  81018               2018-05-09 15:48:00   
order_delivered_customer_date  96476  95664               2018-05-14 20:02:44   
order_estimated_delivery_date  99441

In [12]:
print("===== ORDER ITEMS =====")
print(order_items[["price", "freight_value"]].describe())

print("\n===== PAYMENTS =====")
print(payments[["payment_value", "payment_installments"]].describe())

print("\n===== REVIEWS =====")
print(reviews[["review_score"]].describe())

print("\n===== PRODUCTS =====")
print(products[[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].describe())

===== ORDER ITEMS =====
               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000

===== PAYMENTS =====
       payment_value  payment_installments
count  103886.000000         103886.000000
mean      154.100380              2.853349
std       217.494064              2.687051
min         0.000000              0.000000
25%        56.790000              1.000000
50%       100.000000              1.000000
75%       171.837500              4.000000
max     13664.080000             24.000000

===== REVIEWS =====
       review_score
count  99224.000000
mean       4.086421
std        1.347579
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000

===== PRODUCTS =====
       product_weigh

In [13]:
print("===== REVIEW SCORE =====")
print(reviews["review_score"].describe())

print("\n===== PRODUCT DIMENSIONS =====")
print(products[[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].describe())

===== REVIEW SCORE =====
count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

===== PRODUCT DIMENSIONS =====
       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000           7.000000           2.000000   
25%          300.000000          18.000000           8.000000   
50%          700.000000          25.000000          13.000000   
75%         1900.000000          38.000000          21.000000   
max        40425.000000         105.000000         105.000000   

       product_width_cm  
count      32949.000000  
mean          23.196728  
std           12.079047  
min            6.000000  
25%           15